# BayMorph — Multi-Observation Joint Fit

This notebook demonstrates joint Bayesian inference from multiple
user-supplied observations of the same astronomical object.

BayMorph does not distribute observational data. Replace the example
paths, target seeds, and scientific priors before running the workflow.
Set `workspace_directory` to an absolute directory containing your
`data/` directory; BayMorph will write analysis products beneath its
`runs/` directory. Absolute paths keep every generated observation
configuration valid when the joint YAML is opened from another working
directory.
Observations remain distinct measurements on their native detector grids;
parameter sharing is an explicit modelling choice and is never inferred
from observation names.

The labels `g`, `r`, and `i` are illustrative identifiers. They may
denote filters, epochs, instruments, or other distinct observations.


## 1. Define the observations

FITS paths and target-identification seeds are explicit inputs. Replace
every `None` centre with a zero-based `(x, y)` pixel coordinate for the
intended source. These coordinates seed preparation; they do not fix the
fitted centroids.

Set `workspace_directory` to a real absolute path before continuing.
Flux-prior bounds are observation-specific scientific inputs. Declare or
calibrate them for the supplied data rather than copying values from
another target.


In [ ]:
from pathlib import Path

from baymorph import (
    BayMorphRunner,
    make_multi_observation_config,
    make_multi_parameters,
)


workspace_directory = Path(
    "/absolute/path/to/BayMorph_workspace"
).expanduser().resolve()

observations = {
    "g": workspace_directory / "data" / "observed_galaxy_g.fits",
    "r": workspace_directory / "data" / "observed_galaxy_r.fits",
    "i": workspace_directory / "data" / "observed_galaxy_i.fits",
}

target_centers = {
    "g": None,
    "r": None,
    "i": None,
}

flux_prior_bounds = {
    "g": None,
    "r": None,
    "i": None,
}

analysis_name = "observed_galaxy_gri_single_sersic"
model_name = "single_sersic"


## 2. Define the scientific coupling

This template shares axis ratio and position angle as an explicit example
hypothesis. Flux, effective radius, Sérsic index, sky position, and
background remain observation-specific.

Sharing sky position through `ra` and `dec` is appropriate only after
WCS registration and astrometric assumptions have been validated. Sharing
effective radius or Sérsic index is another stronger hypothesis and must
be tested deliberately.

The non-flux bounds below are illustrative starting choices, not universal
defaults. Review every prior against the supplied observations and the
scientific question.


In [ ]:
missing_centres = [
    name
    for name, centre in target_centers.items()
    if centre is None
]

if missing_centres:
    raise ValueError(
        "Set an explicit target centre for every observation: "
        f"{missing_centres}."
    )

missing_flux_priors = [
    name
    for name, bounds in flux_prior_bounds.items()
    if bounds is None
]

if missing_flux_priors:
    raise ValueError(
        "Set or calibrate flux-prior bounds for every observation: "
        f"{missing_flux_priors}."
    )

prior_bounds = {
    "flux": flux_prior_bounds,
    "re_arcsec": (0.2, 45.0),
    "n": (0.2, 6.0),
    "q": (0.1, 1.0),
}

shared_parameters = (
    "q",
    "position_angle",
)

parameters = make_multi_parameters(
    model=model_name,
    observations=observations,
    target_centers=target_centers,
    prior_bounds=prior_bounds,
    shared=shared_parameters,
    parameter_frame="sky",
)


## 3. Generate the joint YAML configuration

BayMorph writes one configuration for each observation and one joint
configuration describing the coupled inference problem. This template
uses a normalized Gaussian likelihood and explicitly selects Nautilus,
an evidence-producing backend.

Notebook 01 demonstrates general sampler selection. This joint tutorial
deliberately does not offer emcee yet: managed
`checkpoint_path: auto` currently belongs to the single-observation
real-fit workflow and must not be copied into a joint configuration until
dedicated joint checkpoint integration is validated.


In [ ]:
outputs_root = workspace_directory / "runs"
config_dir = outputs_root / analysis_name / "configs"
joint_config_path = config_dir / "joint.yaml"

joint_config = make_multi_observation_config(
    observations=observations,
    target_positions=target_centers,
    parameters=parameters,
    output_dir=config_dir,
    joint_output_path=joint_config_path,
    analysis_name=analysis_name,
    model_name=model_name,
    parameter_frame="sky",
    likelihood={
        "type": "gaussian",
        "include_normalization": True,
    },
    sampler={
        "name": "nautilus",
        "execution": {
            "processes": 1,
            "progress": True,
        },
        "init": {
            "n_live": 300,
            "seed": 42,
        },
        "run": {
            "n_eff": 500,
            "f_live": 0.02,
        },
    },
    outputs_root=outputs_root,
)

print(f"Joint configuration: {joint_config_path}")

for name, entry in joint_config["observations"].items():
    print(f"{name}: {entry['config_path']}")


## 4. Run the all-in-one workflow

`BayMorphRunner` recognizes the joint configuration and coordinates
preparation, PSF construction, joint inference, persistence, and
diagnostics through the same high-level interface used for one
observation.


In [ ]:
runner = BayMorphRunner(
    joint_config_path,
    verbose=True,
)

result = runner.run(
    diagnostics=True,
)

result.summary()


## 5. Step-by-step equivalent

The all-in-one workflow above is recommended. To inspect each stage in a
fresh Python session:

    runner = BayMorphRunner(joint_config_path)
    runner.prepare_image()
    runner.select_psf_candidates()
    runner.build_psf()
    result = runner.fit()
    runner.corner()
    runner.residual_maps()
    profile_bands = runner.radial_profile()

The global posterior and Bayesian evidence describe the complete joint
probabilistic model selected here. Detector-space models and residuals
remain observation-specific.

In sky-frame inference, BayMorph maps sky-coordinate parameters through
each observation's WCS rather than resampling images onto a shared grid.
This template keeps sky position independent; sharing it requires
validated astrometric registration.

Changing which parameters are shared changes the scientific model and
therefore the interpretation of its evidence.
